# Notebook 4: Confidence Intervals and Bootstrap Methods

## Overview
In this notebook, we explore **confidence intervals** and **bootstrap resampling**—two powerful tools for quantifying uncertainty in A/B testing results.

### Learning Objectives
- Understand what confidence intervals really mean (and common misconceptions)
- Calculate confidence intervals analytically for proportions and means
- Learn the bootstrap principle: resampling to estimate uncertainty
- Implement bootstrap hypothesis tests
- Compare analytical vs. bootstrap approaches

### Why This Matters
A/B test results should always include uncertainty estimates. A point estimate alone (e.g., "conversion rate is 5%") is incomplete. Confidence intervals tell us the range of plausible values and help us make rigorous decisions.

**Key Insight**: The bootstrap is a practical, flexible method that works for almost any statistic, even when we don't have a formula for its distribution.

In [1]:
import os
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Try to import plotly for optional interactive plots
try:
    import plotly.express as px
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

# Set style
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (12, 6)

# Create output directory
os.makedirs('../data/outputs/nb04', exist_ok=True)


In [2]:
# Load cleaned data
data_path = Path("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df = pd.read_csv(data_path)

print(f"Data shape: {df.shape}")
print(f"\nFirst rows:\n{df.head()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nGroup distribution:\n{df['segment'].value_counts()}")

Data shape: (64000, 24)

First rows:
   recency history_segment  history  mens  womens   zip_code  newbie channel  \
0       10  2) $100 - $200   142.44     1       0  Surburban       0   Phone   
1        6  3) $200 - $350   329.08     1       1      Rural       1     Web   
2        7  2) $100 - $200   180.65     0       1  Surburban       1     Web   
3        9  5) $500 - $750   675.83     1       0      Rural       1     Web   
4        2    1) $0 - $100    45.34     1       0      Urban       0     Web   

         segment  visit  ...  treatment   buyer_type cross_shopper  \
0  Womens E-Mail      0  ...          1    mens_only             0   
1      No E-Mail      0  ...          0         both             1   
2  Womens E-Mail      0  ...          1  womens_only             0   
3    Mens E-Mail      0  ...          1    mens_only             0   
4  Womens E-Mail      0  ...          1    mens_only             0   

  zip_code_encoded  channel_encoded history_log  spending_vel

## Concept: What Are Confidence Intervals?

### The (Misunderstood) Idea
A **confidence interval** is a range of values computed from sample data, designed to bracket an unknown population parameter.

#### Common Misconception
Many people think a 95% CI means "there's a 95% probability the true parameter is in this interval." **This is wrong!**

The true parameter is either in the interval or it isn't—there's no probability. Instead, 95% CI means:

> If we repeated our experiment many times, ~95% of the intervals we construct would contain the true parameter.

It's a **long-run frequency** property, not a direct probability about this particular interval.

#### Types of Confidence Intervals
1. **Wald Interval** (Normal approximation): Simple, but can be inaccurate for proportions near 0 or 1
2. **Wilson Score Interval**: More accurate, especially for small samples and extreme proportions
3. **Bootstrap Percentile**: Uses resampling, works for any statistic
4. **Highest Density Interval (HDI)**: Bayesian approach (covered in Notebook 6)

#### In A/B Testing
We usually care about:
- Confidence intervals for **conversion rate difference** (e.g., Men's - Control)
- Confidence intervals for **average spend difference**
- Whether the CI excludes zero (suggesting a real difference)

## Analytical Confidence Intervals

### Wald and Wilson Intervals for Proportions
For a single proportion p, we can calculate CIs analytically.
- **Wald CI**: Uses the normal approximation; simple but unreliable for extreme p
- **Wilson CI**: Uses score inversion; more robust

For the **difference** between two proportions, we typically use the Wald method with a continuity correction.

In [3]:
def wald_ci_proportion(successes, n, confidence=0.95):
    """
    Calculate Wald confidence interval for a proportion.
    
    Parameters:
    -----------
    successes : int
        Number of successes
    n : int
        Total sample size
    confidence : float
        Confidence level (default 0.95)
    
    Returns:
    --------
    tuple : (point_estimate, lower_bound, upper_bound)
    """
    p = successes / n
    se = np.sqrt(p * (1 - p) / n)
    z = stats.norm.ppf((1 + confidence) / 2)
    
    lower = p - z * se
    upper = p + z * se
    
    return p, max(0, lower), min(1, upper)

def wilson_ci_proportion(successes, n, confidence=0.95):
    """
    Calculate Wilson score confidence interval for a proportion.
    More robust than Wald, especially for extreme proportions.
    """
    p = successes / n
    z = stats.norm.ppf((1 + confidence) / 2)
    z_sq = z ** 2
    
    denom = 1 + z_sq / n
    center = (p + z_sq / (2 * n)) / denom
    margin = z * np.sqrt(p * (1 - p) / n + z_sq / (4 * n ** 2)) / denom
    
    return p, center - margin, center + margin

def ci_difference_proportions(succ1, n1, succ2, n2, confidence=0.95):
    """
    Wald CI for difference between two proportions: p1 - p2
    """
    p1 = succ1 / n1
    p2 = succ2 / n2
    diff = p1 - p2
    
    se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    z = stats.norm.ppf((1 + confidence) / 2)
    
    return diff, diff - z * se, diff + z * se

# Calculate conversion rate for each segment
segment_stats = df.groupby('segment').agg({
    'conversion': ['sum', 'count', 'mean']
}).round(4)

segment_stats.columns = ['Conversions', 'N', 'Rate']
print("Conversion Statistics by Segment:")
print(segment_stats)
print()

# Calculate CIs for conversion rates
groups = df['segment'].unique()
ci_results = []

for group in groups:
    group_data = df[df['segment'] == group]
    conv_count = group_data['conversion'].sum()
    total = len(group_data)
    
    # Wald CI
    p_wald, lower_wald, upper_wald = wald_ci_proportion(conv_count, total)
    
    # Wilson CI
    p_wilson, lower_wilson, upper_wilson = wilson_ci_proportion(conv_count, total)
    
    ci_results.append({
        'Segment': group,
        'N': total,
        'Conversions': conv_count,
        'Rate': p_wald,
        'Wald Lower': lower_wald,
        'Wald Upper': upper_wald,
        'Wilson Lower': lower_wilson,
        'Wilson Upper': upper_wilson
    })

ci_df = pd.DataFrame(ci_results)
print("Confidence Intervals for Conversion Rates (95%):")
print(ci_df.to_string(index=False))

Conversion Statistics by Segment:
               Conversions      N    Rate
segment                                  
Mens E-Mail            267  21307  0.0125
No E-Mail              122  21306  0.0057
Womens E-Mail          189  21387  0.0088

Confidence Intervals for Conversion Rates (95%):
      Segment     N  Conversions     Rate  Wald Lower  Wald Upper  Wilson Lower  Wilson Upper
Womens E-Mail 21387          189 0.008837    0.007583    0.010091      0.007668      0.010183
    No E-Mail 21306          122 0.005726    0.004713    0.006739      0.004798      0.006832
  Mens E-Mail 21307          267 0.012531    0.011037    0.014025      0.011123      0.014115


In [4]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Concept: The Bootstrap

### The Big Idea
The **bootstrap** is a resampling method that uses your sample to estimate the distribution of a statistic without making strong assumptions.

### How It Works (Intuitive Example)
Imagine you have a bag of 100 marbles from a factory. You draw a random sample of 10 marbles. The bootstrap asks:

> "If I repeatedly draw samples (with replacement) from these 10 marbles, what can I learn about the distribution of the sample proportion?"

The intuition: **the sample is a mini-population**. By resampling from it, we can estimate the sampling distribution of our statistic.

### The Algorithm (Percentile Bootstrap)
1. **Original sample**: Your data
2. **Resample**: Draw n observations **with replacement** from your data (n times)
3. **Calculate statistic**: Compute your statistic (mean, median, difference, etc.) on the resample
4. **Repeat steps 2-3** many times (typically 10,000+)
5. **Percentiles**: Use the 2.5th and 97.5th percentiles of the bootstrap distribution as 95% CI

### Why It Works
The bootstrap relies on a key insight: if your sample is representative of the population, then the sample is representative of the population in the same way the bootstrap resamples are representative of the sample.

### Advantages
- **Works for any statistic**: mean, median, ratio, difference, correlation, etc.
- **No distributional assumptions**: doesn't assume normality
- **Handles small samples**: sometimes more accurate than theory-based methods
- **Flexible**: can implement complex sampling designs, weighted samples, etc.

### Disadvantages
- **Computationally expensive** (though usually fast for modern computers)
- **Requires representative sample**: if the sample is biased, bootstrap results are biased
- **Works poorly with extreme statistics** (e.g., max of a small sample)

In [5]:
def bootstrap_diff_conversion(df, group1_name, group2_name, n_bootstrap=10000, random_state=42):
    """
    Bootstrap confidence interval for difference in conversion rates: group1 - group2
    
    Parameters:
    -----------
    df : DataFrame
        Data with 'segment' and 'conversion' columns
    group1_name : str
        Name of first group
    group2_name : str
        Name of second group (comparison baseline)
    n_bootstrap : int
        Number of bootstrap samples
    random_state : int
        Random seed
    
    Returns:
    --------
    dict : Contains bootstrap differences, CI, and statistics
    """
    np.random.seed(random_state)
    
    # Get data for each group
    conv1 = df[df['segment'] == group1_name]['conversion'].values
    conv2 = df[df['segment'] == group2_name]['conversion'].values
    
    # Original difference
    original_diff = conv1.mean() - conv2.mean()
    
    # Bootstrap loop
    boot_diffs = []
    for _ in range(n_bootstrap):
        # Resample with replacement
        boot_conv1 = np.random.choice(conv1, size=len(conv1), replace=True)
        boot_conv2 = np.random.choice(conv2, size=len(conv2), replace=True)
        
        boot_diffs.append(boot_conv1.mean() - boot_conv2.mean())
    
    boot_diffs = np.array(boot_diffs)
    
    # Calculate percentile CI (2.5th to 97.5th)
    ci_lower = np.percentile(boot_diffs, 2.5)
    ci_upper = np.percentile(boot_diffs, 97.5)
    
    # Calculate BCa (Bias-Corrected and Accelerated) CI
    # Bias correction
    z0 = stats.norm.ppf(np.mean(boot_diffs < original_diff))
    
    # Acceleration (jackknife)
    jack_diffs = []
    for i in range(len(conv1)):
        conv1_jack = np.delete(conv1, i)
        conv2_sample = conv2.copy()
        jack_diffs.append(conv1_jack.mean() - conv2_sample.mean())
    
    for i in range(len(conv2)):
        conv1_sample = conv1.copy()
        conv2_jack = np.delete(conv2, i)
        jack_diffs.append(conv1_sample.mean() - conv2_jack.mean())
    
    jack_diffs = np.array(jack_diffs)
    jack_mean = np.mean(jack_diffs)
    
    num = np.sum((jack_mean - jack_diffs) ** 3)
    denom = 6 * (np.sum((jack_mean - jack_diffs) ** 2) ** 1.5)
    accel = num / denom if denom != 0 else 0
    
    # BCa percentiles
    p_lower = stats.norm.cdf(z0 + (z0 + stats.norm.ppf(0.025)) / (1 - accel * (z0 + stats.norm.ppf(0.025))))
    p_upper = stats.norm.cdf(z0 + (z0 + stats.norm.ppf(0.975)) / (1 - accel * (z0 + stats.norm.ppf(0.975))))
    
    bca_lower = np.percentile(boot_diffs, p_lower * 100)
    bca_upper = np.percentile(boot_diffs, p_upper * 100)
    
    return {
        'group1': group1_name,
        'group2': group2_name,
        'original_diff': original_diff,
        'boot_differences': boot_diffs,
        'percentile_ci': (ci_lower, ci_upper),
        'bca_ci': (bca_lower, bca_upper),
        'se': np.std(boot_diffs),
        'n_group1': len(conv1),
        'n_group2': len(conv2)
    }

# Run bootstrap for Men's vs Control and Women's vs Control
print("Running bootstrap analysis...")
boot_mens = bootstrap_diff_conversion(df, "Mens E-Mail", "No E-Mail", n_bootstrap=10000)
boot_womens = bootstrap_diff_conversion(df, "Womens E-Mail", "No E-Mail", n_bootstrap=10000)

print("\n=== Bootstrap Results: Conversion Rate Difference ===\n")

for boot_result in [boot_mens, boot_womens]:
    print(f"{boot_result['group1']} vs {boot_result['group2']}:")
    print(f"  Original difference: {boot_result['original_diff']:.4f}")
    print(f"  Bootstrap SE: {boot_result['se']:.4f}")
    print(f"  Percentile 95% CI: [{boot_result['percentile_ci'][0]:.4f}, {boot_result['percentile_ci'][1]:.4f}]")
    print(f"  BCa 95% CI:        [{boot_result['bca_ci'][0]:.4f}, {boot_result['bca_ci'][1]:.4f}]")
    print()

Running bootstrap analysis...

=== Bootstrap Results: Conversion Rate Difference ===

Mens E-Mail vs No E-Mail:
  Original difference: 0.0068
  Bootstrap SE: 0.0009
  Percentile 95% CI: [0.0050, 0.0086]
  BCa 95% CI:        [0.0050, 0.0086]

Womens E-Mail vs No E-Mail:
  Original difference: 0.0031
  Bootstrap SE: 0.0008
  Percentile 95% CI: [0.0015, 0.0047]
  BCa 95% CI:        [0.0015, 0.0047]



In [6]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Bootstrap for Continuous Outcomes: Spend Difference

The bootstrap works equally well for continuous outcomes. Instead of resampling conversion (binary), we resample spend values.

In [7]:
def bootstrap_diff_continuous(df, metric, group1_name, group2_name, n_bootstrap=10000, random_state=42):
    """
    Bootstrap confidence interval for difference in a continuous metric.
    
    Parameters:
    -----------
    df : DataFrame
        Data with 'segment' column
    metric : str
        Column name of continuous metric (e.g., 'spend')
    group1_name : str
        Name of first group
    group2_name : str
        Name of second group
    n_bootstrap : int
        Number of bootstrap samples
    random_state : int
        Random seed
    
    Returns:
    --------
    dict : Bootstrap results
    """
    np.random.seed(random_state)
    
    # Get metric values for each group
    metric1 = df[df['segment'] == group1_name][metric].values
    metric2 = df[df['segment'] == group2_name][metric].values
    
    # Original difference
    original_diff = metric1.mean() - metric2.mean()
    
    # Bootstrap loop
    boot_diffs = []
    for _ in range(n_bootstrap):
        boot_metric1 = np.random.choice(metric1, size=len(metric1), replace=True)
        boot_metric2 = np.random.choice(metric2, size=len(metric2), replace=True)
        boot_diffs.append(boot_metric1.mean() - boot_metric2.mean())
    
    boot_diffs = np.array(boot_diffs)
    
    ci_lower = np.percentile(boot_diffs, 2.5)
    ci_upper = np.percentile(boot_diffs, 97.5)
    
    return {
        'group1': group1_name,
        'group2': group2_name,
        'metric': metric,
        'original_diff': original_diff,
        'boot_differences': boot_diffs,
        'ci': (ci_lower, ci_upper),
        'se': np.std(boot_diffs),
        'n_group1': len(metric1),
        'n_group2': len(metric2)
    }

# Run bootstrap for spend
print("Running bootstrap for spend difference...\n")
boot_spend_mens = bootstrap_diff_continuous(df, 'spend', "Mens E-Mail", "No E-Mail", n_bootstrap=10000)
boot_spend_womens = bootstrap_diff_continuous(df, 'spend', "Womens E-Mail", "No E-Mail", n_bootstrap=10000)

for boot_result in [boot_spend_mens, boot_spend_womens]:
    print(f"{boot_result['group1']} vs {boot_result['group2']} - Spend:")
    print(f"  Original difference: ${boot_result['original_diff']:.2f}")
    print(f"  Bootstrap SE: ${boot_result['se']:.2f}")
    print(f"  95% CI: [${boot_result['ci'][0]:.2f}, ${boot_result['ci'][1]:.2f}]")
    print()

Running bootstrap for spend difference...

Mens E-Mail vs No E-Mail - Spend:
  Original difference: $0.77
  Bootstrap SE: $0.14
  95% CI: [$0.49, $1.06]

Womens E-Mail vs No E-Mail - Spend:
  Original difference: $0.42
  Bootstrap SE: $0.13
  95% CI: [$0.18, $0.69]



In [8]:
# (Chart cell removed — interactive Plotly version rendered below in the "Blog-Ready Plotly Charts" section.)

## Bootstrap Hypothesis Test

### The Idea
Instead of computing a test statistic under a null hypothesis (like frequentist methods), the bootstrap hypothesis test computes:

> **p-value = proportion of bootstrap samples where the statistic is as extreme as observed**

For testing H₀: difference = 0, we ask: "If there were no difference, how often would we see a difference this large just by chance in resamples?"

### Two Approaches
1. **Percentile p-value**: Does zero fall in the 95% CI?
2. **Permutation bootstrap**: Resample from the combined data under H₀ to get the null distribution

Here we'll use the simpler percentile approach.

In [9]:
def bootstrap_p_value(boot_differences, null_value=0):
    """
    Calculate bootstrap p-value for a two-tailed test.
    
    Parameters:
    -----------
    boot_differences : array
        Bootstrap resamples of the statistic
    null_value : float
        Hypothesized value under H₀ (default 0)
    
    Returns:
    --------
    float : Two-tailed p-value
    """
    centered = boot_differences - np.mean(boot_differences) + null_value
    p_value = np.mean(np.abs(centered - null_value) >= np.abs(np.mean(boot_differences) - null_value))
    return p_value

print("=== Bootstrap Hypothesis Tests (H₀: difference = 0) ===\n")

for boot_result in [boot_mens, boot_womens]:
    p_val = bootstrap_p_value(boot_result['boot_differences'])
    print(f"{boot_result['group1']} vs {boot_result['group2']}:")
    print(f"  Bootstrap p-value: {p_val:.4f}")
    ci_lower, ci_upper = boot_result['percentile_ci']
    print(f"  CI contains zero: {ci_lower <= 0 <= ci_upper}")
    print()

print("\nInterpretation:")
print("- p < 0.05: Reject H₀, likely a real difference")
print("- p >= 0.05: Fail to reject H₀, no strong evidence of difference")

=== Bootstrap Hypothesis Tests (H₀: difference = 0) ===

Mens E-Mail vs No E-Mail:
  Bootstrap p-value: 0.0000
  CI contains zero: False

Womens E-Mail vs No E-Mail:
  Bootstrap p-value: 0.0000
  CI contains zero: False


Interpretation:
- p < 0.05: Reject H₀, likely a real difference
- p >= 0.05: Fail to reject H₀, no strong evidence of difference


## Comparison: Analytical vs Bootstrap CIs

### Side-by-Side Comparison
Let's compare the analytical Wald CI with bootstrap percentile CI. Both should be similar for conversion rates on large samples.

In [10]:
# Compile comparison table
comparison_data = []

for boot_result in [boot_mens, boot_womens]:
    group1, group2 = boot_result['group1'], boot_result['group2']
    
    # Get analytical CI
    g1_data = df[df['segment'] == group1]
    g2_data = df[df['segment'] == group2]
    
    conv1_count = g1_data['conversion'].sum()
    conv2_count = g2_data['conversion'].sum()
    n1, n2 = len(g1_data), len(g2_data)
    
    analytical_diff, analytical_lower, analytical_upper = ci_difference_proportions(
        conv1_count, n1, conv2_count, n2
    )
    
    # Bootstrap CI
    boot_lower, boot_upper = boot_result['percentile_ci']
    
    comparison_data.append({
        'Comparison': f"{group1} vs {group2}",
        'Observed Diff': f"{boot_result['original_diff']:.4f}",
        'Analytical CI': f"[{analytical_lower:.4f}, {analytical_upper:.4f}]",
        'Bootstrap CI': f"[{boot_lower:.4f}, {boot_upper:.4f}]",
        'Width (Analytical)': f"{analytical_upper - analytical_lower:.4f}",
        'Width (Bootstrap)': f"{boot_upper - boot_lower:.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n=== Analytical vs Bootstrap Confidence Intervals ===\n")
print(comparison_df.to_string(index=False))

# Save comparison
comparison_df.to_csv('../data/outputs/nb04/nb04_ci_comparison.csv', index=False)


=== Analytical vs Bootstrap Confidence Intervals ===

                Comparison Observed Diff    Analytical CI     Bootstrap CI Width (Analytical) Width (Bootstrap)
  Mens E-Mail vs No E-Mail        0.0068 [0.0050, 0.0086] [0.0050, 0.0086]             0.0036            0.0036
Womens E-Mail vs No E-Mail        0.0031 [0.0015, 0.0047] [0.0015, 0.0047]             0.0032            0.0032


## When Bootstrap is Better than Analytical Methods

### Small Sample Sizes
- Analytical methods assume normality, which may not hold with small n
- Bootstrap gives accurate results without distributional assumptions
- **Example**: Testing on only 50 customers per group

### Non-Normal Data
- Spend data is often skewed (many small purchases, few large ones)
- Bootstrap works on the actual distribution of your data
- Analytical methods assume normality and may be inaccurate

### Complex Statistics
- Analytical formulas don't exist for many statistics (e.g., median, trimmed mean, ratio)
- Bootstrap can estimate the sampling distribution of **any** statistic
- **Example**: Bootstrap CI for 90th percentile spend

### Weighted or Stratified Samples
- Bootstrap easily adapts to complex sampling designs
- Analytical methods require mathematical rework

### Robustness
- Bootstrap is more robust to violations of assumptions
- Always gives reasonable results (though maybe not perfectly calibrated)

### Computational Cost
- Modern computers can do 10,000 bootstrap resamples in milliseconds
- The only real drawback is communicating uncertainty to non-technical audiences

### Best Practice
Use both analytical and bootstrap methods. Agreement suggests robust results; disagreement warrants investigation.

In [11]:
# Create comprehensive results summary
results_summary = {
    'Metric': ['Conversion Rate - Men\'s vs Control', 'Conversion Rate - Women\'s vs Control',
               'Spend - Men\'s vs Control', 'Spend - Women\'s vs Control'],
    'Observed Difference': [
        f"{boot_mens['original_diff']:.4f}",
        f"{boot_womens['original_diff']:.4f}",
        f"${boot_spend_mens['original_diff']:.2f}",
        f"${boot_spend_womens['original_diff']:.2f}"
    ],
    '95% Bootstrap CI': [
        f"[{boot_mens['percentile_ci'][0]:.4f}, {boot_mens['percentile_ci'][1]:.4f}]",
        f"[{boot_womens['percentile_ci'][0]:.4f}, {boot_womens['percentile_ci'][1]:.4f}]",
        f"[${boot_spend_mens['ci'][0]:.2f}, ${boot_spend_mens['ci'][1]:.2f}]",
        f"[${boot_spend_womens['ci'][0]:.2f}, ${boot_spend_womens['ci'][1]:.2f}]"
    ],
    'Includes Zero/No Effect': [
        'Yes' if boot_mens['percentile_ci'][0] <= 0 <= boot_mens['percentile_ci'][1] else 'No',
        'Yes' if boot_womens['percentile_ci'][0] <= 0 <= boot_womens['percentile_ci'][1] else 'No',
        'Yes' if boot_spend_mens['ci'][0] <= 0 <= boot_spend_mens['ci'][1] else 'No',
        'Yes' if boot_spend_womens['ci'][0] <= 0 <= boot_spend_womens['ci'][1] else 'No'
    ]
}

summary_df = pd.DataFrame(results_summary)
print("\n=== BOOTSTRAP ANALYSIS SUMMARY ===\n")
print(summary_df.to_string(index=False))

# Save detailed results
summary_df.to_csv('../data/outputs/nb04/nb04_bootstrap_results.csv', index=False)
print("\nResults saved to ../data/outputs/nb04/nb04_bootstrap_results.csv")


=== BOOTSTRAP ANALYSIS SUMMARY ===

                              Metric Observed Difference 95% Bootstrap CI Includes Zero/No Effect
  Conversion Rate - Men's vs Control              0.0068 [0.0050, 0.0086]                      No
Conversion Rate - Women's vs Control              0.0031 [0.0015, 0.0047]                      No
            Spend - Men's vs Control               $0.77   [$0.49, $1.06]                      No
          Spend - Women's vs Control               $0.42   [$0.18, $0.69]                      No

Results saved to ../data/outputs/nb04/nb04_bootstrap_results.csv


---

## Blog-Ready Plotly Charts

The cells below regenerate the charts from this notebook as responsive Plotly
HTML files for embedding in the blog post. They are **self-contained**: each
one re-loads the clean dataset from nb01 and re-derives the statistics it
needs, so you can run this section in isolation.

Outputs are written to `data/outputs/nb##/` with the suffix `_interactive.html`.

**Required packages:** `plotly` (install with `pip install plotly` if missing).

In [12]:
# ============================================================
# Blog-Ready Plotly Charts — self-contained, embed-friendly
# ============================================================
# These cells produce responsive Plotly HTML files for the blog post.
# They re-load from the nb01 clean CSV and re-derive stats so the section
# runs standalone. Each figure uses:
#   - include_plotlyjs='cdn' (single shared CDN load on the blog page)
#   - config={'responsive': True} so it resizes to container width
#   - automargin=True on axes + generous margins so labels never clip
#   - rotated tick labels on long categories, headroom for outside labels
import os, numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook_connected"  # inline-render Plotly in cell output

OUT_DIR = os.path.abspath("../data/outputs/nb04")
os.makedirs(OUT_DIR, exist_ok=True)
CLEAN_CSV = os.path.abspath("../data/outputs/nb01/nb01_hillstrom_clean.csv")
df_blog = pd.read_csv(CLEAN_CSV)

# Shared palette aligned with nb01 Plotly charts
COLORS = {
    "Mens E-Mail": "#4C8BB8", "Womens E-Mail": "#5FA85F", "No E-Mail": "#E89B4C",
    "Match": "#2ECC71", "Mismatch": "#E74C3C", "Mixed": "#F39C12", "Control": "#95A5A6",
    "Treatment (Any Email)": "#4C8BB8",
}
PLOTLY_KW = dict(include_plotlyjs="cdn", full_html=True,
                 config={"responsive": True, "displaylogo": False})
BASE_LAYOUT = dict(template="plotly_white",
                   font=dict(family="Arial, sans-serif", size=13),
                   title_x=0.5,
                   margin=dict(l=70, r=40, t=90, b=90),
                   hoverlabel=dict(bgcolor="white", font_size=12))
print(f"Blog-ready Plotly charts will be written to: {OUT_DIR}")


Blog-ready Plotly charts will be written to: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/ab_testing/data/outputs/nb04


In [15]:
# Chart 1: Bootstrap distribution of conversion-rate difference with CI overlay
np.random.seed(42)
treated = df_blog[df_blog["segment"] != "No E-Mail"]["conversion"].values
control_arr = df_blog[df_blog["segment"] == "No E-Mail"]["conversion"].values
B = 5000
boot_diffs = np.empty(B)
for b in range(B):
    t_s = np.random.choice(treated, size=len(treated), replace=True)
    c_s = np.random.choice(control_arr, size=len(control_arr), replace=True)
    boot_diffs[b] = t_s.mean() - c_s.mean()
pct_lo, pct_hi = np.percentile(boot_diffs, [2.5, 97.5])
obs_diff = treated.mean() - control_arr.mean()

fig = go.Figure()
fig.add_trace(go.Histogram(x=boot_diffs*100, nbinsx=60,
                           marker=dict(color="#4C8BB8", line=dict(color="black", width=0.5)),
                           name="Bootstrap replicates",
                           hovertemplate="Diff: %{x:.3f} pp<br>Count: %{y}<extra></extra>"))
fig.add_vline(x=obs_diff*100, line_color="black", line_width=3,
              annotation_text=f"Observed: {obs_diff*100:+.3f} pp", annotation_position="top")
fig.add_vline(x=pct_lo*100, line_dash="dash", line_color="#E74C3C",
              annotation_text="2.5%", annotation_position="top left")
fig.add_vline(x=pct_hi*100, line_dash="dash", line_color="#E74C3C",
              annotation_text="97.5%", annotation_position="top right")
fig.update_layout(**BASE_LAYOUT,
                  title=f"Bootstrap Distribution — Treatment - Control Conversion<br>"
                        f"<sub>95% CI [{pct_lo*100:+.3f}, {pct_hi*100:+.3f}] pp over {B:,} replicates</sub>",
                  xaxis=dict(title="Conversion-Rate Difference (pp)", automargin=True),
                  yaxis=dict(title="Count", automargin=True), height=500, bargap=0.02)
fig.write_html(os.path.join(OUT_DIR, "nb04_bootstrap_distribution_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb04_bootstrap_distribution_interactive.html")

# Chart 2: Analytical vs Bootstrap CI forest plot
def wald(s, n):
    p = s/n; se = np.sqrt(p*(1-p)/n)
    return p, p - 1.96*se, p + 1.96*se

segments = ["Mens E-Mail", "Womens E-Mail", "No E-Mail"]
rows=[]
for seg in segments:
    g = df_blog[df_blog["segment"]==seg]["conversion"]
    p, lo, hi = wald(g.sum(), len(g))
    rows.append((seg, "Wald (analytical)", p, lo, hi))
    # bootstrap
    vals = g.values; boot=np.empty(2000)
    for b in range(2000):
        boot[b] = np.random.choice(vals, size=len(vals), replace=True).mean()
    blo, bhi = np.percentile(boot, [2.5, 97.5])
    rows.append((seg, "Bootstrap percentile", p, blo, bhi))
dfci = pd.DataFrame(rows, columns=["seg","method","p","lo","hi"])

fig = go.Figure()
labels=[]
for i,r in dfci.reset_index(drop=True).iterrows():
    lbl = f"{r['seg']} — {r['method']}"; labels.append(lbl)
    col = "#4C8BB8" if "Wald" in r["method"] else "#F39C12"
    fig.add_trace(go.Scatter(x=[r["lo"]*100, r["hi"]*100], y=[i,i],
                             mode="lines", line=dict(color=col, width=3),
                             showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=[r["p"]*100], y=[i], mode="markers",
                             marker=dict(size=12, color=col, line=dict(color="black", width=1)),
                             showlegend=False,
                             hovertemplate=f"<b>{lbl}</b><br>Estimate: %{{x:.3f}}%<br>CI: [{r['lo']*100:.3f}, {r['hi']*100:.3f}]%<extra></extra>"))
fig.update_yaxes(tickvals=list(range(len(dfci))), ticktext=labels, automargin=True)
fig.update_xaxes(title="Conversion rate (%)", automargin=True)
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=220, r=40, t=90, b=80)},
                  title="Wald vs Bootstrap 95% CIs per Arm", height=500)
fig.write_html(os.path.join(OUT_DIR, "nb04_ci_comparison_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb04_ci_comparison_interactive.html")


  ✓ nb04_bootstrap_distribution_interactive.html


  ✓ nb04_ci_comparison_interactive.html


### Results-Display Tables (embed-ready go.Table cards for every printed output)

Each card below mirrors one of the printed console blocks and saves as its own
HTML file under `../data/outputs/nb04/` so it can be dropped straight into the
blog post.


In [14]:
# Reusable go.Table card helper (reuses OUT_DIR/PLOTLY_KW/BASE_LAYOUT from Plotly setup cell above)
def table_card(title, header_vals, cell_cols, colwidths,
               row_colors=None, cell_font_size=12, height_extra=80, align="center"):
    n_rows = len(cell_cols[0]) if cell_cols else 0
    stripe = ["#F8F9F9" if i%2==0 else "white" for i in range(n_rows)]
    fill = row_colors if row_colors else [stripe for _ in cell_cols]
    fig = go.Figure(data=[go.Table(
        columnwidth=colwidths,
        header=dict(values=[f"<b>{h}</b>" for h in header_vals],
                    fill_color="#2C3E50",
                    font=dict(color="white", size=13),
                    align="center", height=36),
        cells=dict(values=cell_cols, fill_color=fill, align=align,
                   font=dict(size=cell_font_size, family="monospace"),
                   height=30))])
    fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                      title=title,
                      height=36 + 30*n_rows + height_extra)
    return fig

def sig_colors(flags):
    return ["#D5F5E3" if f else "#FADBD8" for f in flags]

def stripe_col(n):
    return ["#F8F9F9" if i%2==0 else "white" for i in range(n)]

# Bootstrap CIs — Results-Display Tables
from scipy.stats import norm as _norm

rng = np.random.default_rng(42)
any_email = df_blog[df_blog["segment"] != "No E-Mail"]
control   = df_blog[df_blog["segment"] == "No E-Mail"]
mens      = df_blog[df_blog["segment"] == "Mens E-Mail"]
womens    = df_blog[df_blog["segment"] == "Womens E-Mail"]

def bootstrap_diff(a, b, stat=np.mean, B=2000):
    a = np.asarray(a); b = np.asarray(b)
    diffs = np.empty(B)
    for i in range(B):
        diffs[i] = stat(rng.choice(a, size=len(a), replace=True)) - \
                   stat(rng.choice(b, size=len(b), replace=True))
    return diffs

def ci_percentile(arr, alpha=0.05):
    return np.percentile(arr, 100*alpha/2), np.percentile(arr, 100*(1-alpha/2))

def wald_diff_prop(p1, n1, p2, n2, alpha=0.05):
    se = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    z = _norm.ppf(1-alpha/2)
    d = p1 - p2
    return d - z*se, d + z*se, d, se

# Visit
pv_e = any_email["visit"].mean(); pv_c = control["visit"].mean()
analytic_lo, analytic_hi, d_v, se_v = wald_diff_prop(pv_e, len(any_email), pv_c, len(control))
bdiffs_v = bootstrap_diff(any_email["visit"].values, control["visit"].values, np.mean, B=2000)
boot_lo_v, boot_hi_v = ci_percentile(bdiffs_v)

# Conversion
pc_e = any_email["conversion"].mean(); pc_c = control["conversion"].mean()
a_lo_c, a_hi_c, d_c, se_c = wald_diff_prop(pc_e, len(any_email), pc_c, len(control))
bdiffs_c = bootstrap_diff(any_email["conversion"].values, control["conversion"].values, np.mean, B=2000)
boot_lo_c, boot_hi_c = ci_percentile(bdiffs_c)

# Spend (Welch-style bootstrap of mean diff)
bdiffs_s = bootstrap_diff(any_email["spend"].values, control["spend"].values, np.mean, B=2000)
boot_lo_s, boot_hi_s = ci_percentile(bdiffs_s)
d_s = any_email["spend"].mean() - control["spend"].mean()
se_s = np.sqrt(any_email["spend"].var(ddof=1)/len(any_email) + control["spend"].var(ddof=1)/len(control))
a_lo_s, a_hi_s = d_s - 1.96*se_s, d_s + 1.96*se_s

# Card 1: Analytical vs Bootstrap comparison
rows = [
    ("Visit rate",     f"{d_v:+.4f}", f"[{analytic_lo:+.4f}, {analytic_hi:+.4f}]", f"[{boot_lo_v:+.4f}, {boot_hi_v:+.4f}]"),
    ("Conversion rate",f"{d_c:+.4f}", f"[{a_lo_c:+.4f}, {a_hi_c:+.4f}]",           f"[{boot_lo_c:+.4f}, {boot_hi_c:+.4f}]"),
    ("Average spend",  f"${d_s:+.4f}", f"[${a_lo_s:+.4f}, ${a_hi_s:+.4f}]",         f"[${boot_lo_s:+.4f}, ${boot_hi_s:+.4f}]"),
]
fig = table_card("Analytical vs Bootstrap 95% Confidence Intervals — Any Email vs Control",
                 ["Outcome", "Point estimate", "Analytical CI (Wald)", "Bootstrap CI (percentile, B=2000)"],
                 [[r[0] for r in rows], [r[1] for r in rows], [r[2] for r in rows], [r[3] for r in rows]],
                 [180, 140, 260, 320])
fig.write_html(os.path.join(OUT_DIR, "nb04_analytical_vs_bootstrap_interactive.html"), **PLOTLY_KW)
fig.show()

# Card 2: Bootstrap summary per outcome (n_replicates, mean, SE, CI width)
def summarize(diffs, name):
    return (name, f"{diffs.mean():+.5f}", f"{diffs.std(ddof=1):.5f}",
            f"[{np.percentile(diffs, 2.5):+.5f}, {np.percentile(diffs, 97.5):+.5f}]",
            f"{np.percentile(diffs, 97.5)-np.percentile(diffs, 2.5):.5f}",
            "Yes" if (np.percentile(diffs,2.5) > 0 or np.percentile(diffs,97.5) < 0) else "No")

summ = [summarize(bdiffs_v, "Visit rate"),
        summarize(bdiffs_c, "Conversion rate"),
        summarize(bdiffs_s, "Average spend")]
sig_flags = [s[-1]=="Yes" for s in summ]
fig = go.Figure(data=[go.Table(
    columnwidth=[160, 140, 120, 260, 140, 130],
    header=dict(values=[f"<b>{h}</b>" for h in
                        ["Outcome","Bootstrap mean","SE","95% Percentile CI","CI width","Significant (0 excluded)?"]],
                fill_color="#2C3E50", font=dict(color="white", size=13), align="center", height=36),
    cells=dict(values=list(zip(*summ)),
               fill_color=[stripe_col(len(summ))]*5 + [sig_colors(sig_flags)],
               align="center", font=dict(size=12, family="monospace"), height=30))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="Bootstrap Analysis Summary (B = 2000 replicates)",
                  height=36 + 30*len(summ) + 80)
fig.write_html(os.path.join(OUT_DIR, "nb04_bootstrap_summary_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb04 analytical-vs-bootstrap + bootstrap summary cards saved")

# -------------------------------------------------------------------
# Per-segment conversion statistics + Wald & Wilson confidence intervals
# -------------------------------------------------------------------
from scipy.stats import norm as _norm_seg

SEGMENT_ORDER = ["Mens E-Mail", "No E-Mail", "Womens E-Mail"]

def wald_ci(p, n, alpha=0.05):
    z = _norm_seg.ppf(1 - alpha/2)
    se = np.sqrt(p*(1-p)/n) if n>0 else 0
    return p - z*se, p + z*se

def wilson_ci(p, n, alpha=0.05):
    if n == 0:
        return 0.0, 0.0
    z = _norm_seg.ppf(1 - alpha/2)
    denom = 1 + z**2/n
    centre = (p + z**2/(2*n)) / denom
    half = (z*np.sqrt(p*(1-p)/n + z**2/(4*n**2))) / denom
    return centre - half, centre + half

# Card A — Conversion statistics by segment (counts, N, rate)
rows = []
for seg in SEGMENT_ORDER:
    g = df_blog[df_blog["segment"] == seg]
    conv = int(g["conversion"].sum())
    n    = len(g)
    rate = conv / n
    rows.append((seg, f"{conv:,}", f"{n:,}", f"{rate:.4f}", f"{rate*100:.2f}%"))

fig = table_card(
    "Conversion Statistics by Segment",
    ["Segment", "Conversions", "N", "Rate", "Rate (%)"],
    [[r[0] for r in rows], [r[1] for r in rows], [r[2] for r in rows],
     [r[3] for r in rows], [r[4] for r in rows]],
    [200, 150, 140, 130, 140])
fig.write_html(os.path.join(OUT_DIR, "nb04_conversion_stats_by_segment_interactive.html"), **PLOTLY_KW)
fig.show()

# Card B — Wald vs Wilson 95% confidence intervals per segment
rows = []
for seg in SEGMENT_ORDER:
    g = df_blog[df_blog["segment"] == seg]
    conv = int(g["conversion"].sum()); n = len(g); p = conv/n
    wl_lo, wl_hi = wald_ci(p, n)
    ws_lo, ws_hi = wilson_ci(p, n)
    rows.append((seg, f"{n:,}", f"{conv:,}", f"{p:.6f}",
                 f"{wl_lo:.6f}", f"{wl_hi:.6f}",
                 f"{ws_lo:.6f}", f"{ws_hi:.6f}",
                 f"[{wl_lo*100:.3f}%, {wl_hi*100:.3f}%]",
                 f"[{ws_lo*100:.3f}%, {ws_hi*100:.3f}%]"))

fig = go.Figure(data=[go.Table(
    columnwidth=[160, 100, 120, 120, 120, 120, 130, 130, 220, 220],
    header=dict(values=[f"<b>{h}</b>" for h in
                        ["Segment","N","Conversions","Rate",
                         "Wald Lower","Wald Upper","Wilson Lower","Wilson Upper",
                         "Wald 95% CI","Wilson 95% CI"]],
                fill_color="#2C3E50", font=dict(color="white", size=13),
                align="center", height=36),
    cells=dict(values=list(zip(*rows)),
               fill_color=[stripe_col(len(rows))]*10,
               align="center", font=dict(size=12, family="monospace"),
               height=30))])
fig.update_layout(**{**BASE_LAYOUT, "margin": dict(l=20, r=20, t=60, b=20)},
                  title="Confidence Intervals for Conversion Rates — Wald vs Wilson (95%)",
                  height=36 + 30*len(rows) + 100)
fig.write_html(os.path.join(OUT_DIR, "nb04_wald_wilson_ci_interactive.html"), **PLOTLY_KW)
fig.show()
print("  ✓ nb04 per-segment conversion stats + Wald/Wilson CI cards saved")


  ✓ nb04 analytical-vs-bootstrap + bootstrap summary cards saved


  ✓ nb04 per-segment conversion stats + Wald/Wilson CI cards saved
